# ASD 解析パイプライン

このチュートリアルでは、`SegmentTable` を使用して、複数セグメントにわたる一括 ASD（振幅スペクトル密度）解析パイプラインを構築する方法を学びます。各セグメントの切り出し（crop）、ASD計算、可視化までを流れるように実行します。

In [ ]:
import numpy as np
from gwpy.segments import Segment
from gwexpy.table import SegmentTable
from gwpy.timeseries import TimeSeries

def get_synthetic_data():
    return TimeSeries(np.random.randn(1024), sample_rate=64)

segs = [Segment(i*16, i*16+16) for i in range(4)]
st = SegmentTable.from_segments(segs)
st.add_series_column("raw", data=[get_synthetic_data()]*len(st), kind="timeseries")
st

## 切り出し (Crop) と ASD 計算

`crop()` と `asd()` という便利な Sugar API を使って、全セグメントを一括処理できます。

In [ ]:
# 各行の期間 (span) でデータを切り出し
st_cropped = st.crop("raw", out_col="cropped")
# セグメントごとに ASD を計算
st_asd = st_cropped.asd("cropped", out_col="asd", fftlength=2.0)
st_asd.display()

## 解析結果の集計

`map()` を使うことで、独自関数（例：特定の周波数帯域の RMS 計算）を適用してテーブルに結果を統合できます。

In [ ]:
def calc_rms(fs):
    return np.sqrt(np.sum(fs.value**2)) # 簡略化した例

st_asd.map("asd", calc_rms, out_col="band_rms", inplace=True)
st_asd.display()